<a href="https://colab.research.google.com/github/Harika143258/Network-Intrusion-Detection-System/blob/main/Network_Intrusion_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TASK-4 NETWORK INTRUSION DETECTION SYSYTEM

In [ ]:

import os
import time
import subprocess
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

def print_banner(title):
    print("\n" + "="*80)
    print(f" >>> {title} <<<")
    print("="*80)

# STEP 1: PROVISION SURICATA IDS INFRASTRUCTURE & RECON TOOLS

print_banner("STEP 1: INSTALLING SURICATA NIDS ENGINE AND DEPENDENCIES")

print("[*] Syncing package managers and compiling system tools...")
subprocess.run(["apt-get", "update", "-qq"], check=True)

# ADDED 'iputils-ping' here to explicitly resolve the FileNotFoundError
subprocess.run(["apt-get", "install", "suricata", "curl", "tcpreplay", "iputils-ping", "-y", "-qq"], check=True)

# Verify deployment integrity
version_check = subprocess.run(["suricata", "-V"], capture_output=True, text=True)
print(f"[+] Engine core online: {version_check.stdout.strip()}")

# STEP 2: CONFIGURE RULESETS AND ENGINE SETTINGS

print_banner("STEP 2: INJECTING MALICIOUS THREAT DETECTION RULES")

RULES_PATH = "/etc/suricata/rules/local.rules"
os.makedirs(os.path.dirname(RULES_PATH), exist_ok=True)

custom_rules = """
# RULE 01: SQL Injection Reconnaissance Pattern
alert http any any -> any any (msg:"ATTACK-RESPONSE SQLi Attempt - SELECT Statement Detected"; content:"select"; nocase; sid:1000001; rev:1;)

# RULE 02: Cross-Site Scripting (XSS) String Injection
alert http any any -> any any (msg:"EXPLOIT-XSS Script Tag Injection Vector Identified"; content:"<script>"; nocase; sid:1000002; rev:1;)

# RULE 03: Unauthorized Corporate Asset Enumeration
alert http any any -> any any (msg:"RECON-PROBE Sensitive Path Browsing - /etc/passwd Access"; content:"/etc/passwd"; sid:1000003; rev:1;)

# RULE 04: Anomalous Network Request Volume / Ping Sweep
alert icmp any any -> any any (msg:"RECON-ICMP Network Mapping Ping Probe Detected"; icmptype:8; sid:1000004; rev:1;)
"""


with open(RULES_PATH, "w") as rules_file:
    rules_file.write(custom_rules.strip())

print(f"[+] Custom rule signatures successfully written to: {RULES_PATH}")

CONFIG_PATH = "/etc/suricata/suricata.yaml"
print("[*] Calibrating configuration matrix paths...")
with open(CONFIG_PATH, "r") as file:
    config_content = file.read()

config_content = config_content.replace("- suricata.rules", "- local.rules")

with open(CONFIG_PATH, "w") as file:
    file.write(config_content)
print("[+] Config engine synced: Active rule validation target locked.")



# STEP 3: INITIALIZE CONTINUOUS NETWORK SENSING (BACKGROUND THREAD)

print_banner("STEP 3: INITIALIZING MOTOR INTERFACE FOR TRAFFIC SENSING")

LOG_DIR = "/var/log/suricata/"
os.makedirs(LOG_DIR, exist_ok=True)

print("[*] Launching Suricata NIDS background runtime daemon tracking loopback interface (lo)...")
suricata_proc = subprocess.Popen(
    ["suricata", "-c", CONFIG_PATH, "-i", "lo", "--init-errors-fatal"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)
print(f"[+] Suricata Engine active under PID: {suricata_proc.pid}. Actively sniffing traffic...")



# STEP 4: INTRODUCE MALICIOUS EXPLOITS (ATTACK SIMULATION)

print_banner("STEP 4: SIMULATING TRAFFIC EXPLOITS & RESPONSE INGESTION")
print("[*] Generating adversarial packet variants over loopback adapter link...\n")

server_proc = subprocess.Popen(["python3", "-m", "http.server", "8080"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

# Vector 1: SQLi
print("[!] Injecting Vector 1: Sending SQL Injection string...")
subprocess.run(["curl", "-s", "http://127.0.0.1:8080/?user=1' UNION SELECT username, password FROM users--"], stdout=subprocess.DEVNULL)

# Vector 2: XSS
print("[!] Injecting Vector 2: Sending XSS Payload...")
subprocess.run(["curl", "-s", "http://127.0.0.1:8080/?q=<script>alert(document.cookie)</script>"], stdout=subprocess.DEVNULL)

# Vector 3: Path Traversal
print("[!] Injecting Vector 3: Accessing Restricted File Paths (/etc/passwd)...")
subprocess.run(["curl", "-s", "http://127.0.0.1:8080/../../etc/passwd"], stdout=subprocess.DEVNULL)

# Vector 4: ICMP Ping Sweep (This will now run perfectly!)
print("[!] Injecting Vector 4: Triggering Ping Probes...")
subprocess.run(["ping", "-c", "4", "127.0.0.1"], stdout=subprocess.DEVNULL)

print("\n[*] Pausing execution sequence briefly to guarantee output telemetry ingestion...")
time.sleep(4)

server_proc.terminate()
suricata_proc.terminate()
print("[+] Network traffic injection phase concluded. Simulation engine deactivated cleanly.")



# STEP 5: VISUALIZATION AND DATA TELEMETRY DASHBOARD

print_banner("STEP 5: COMPILING ANALYTICS DASHBOARD & COMPLIANCE METRICS")

EVE_JSON_PATH = os.path.join(LOG_DIR, "eve.json")

if not os.path.exists(EVE_JSON_PATH) or os.path.getsize(EVE_JSON_PATH) == 0:
    print("[-] Error Matrix: Telemetry ingestion empty. Check interface settings.")
else:
    alert_events = []
    with open(EVE_JSON_PATH, "r") as log_file:
        for log_line in log_file:
            log_record = json.loads(log_line)
            if "alert" in log_record:
                alert_events.append({
                    "Timestamp": log_record.get("timestamp")[:19],
                    "Signature": log_record["alert"].get("signature"),
                    "Category": log_record["alert"].get("category", "Unclassified Threat"),
                    "Severity": log_record["alert"].get("severity", 3),
                    "Src_IP": log_record.get("src_ip"),
                    "Dest_Port": log_record.get("dest_port", "N/A")
                })

    df_alerts = pd.DataFrame(alert_events)

    display(HTML("<h3>🚨 REAL-TIME SYSTEM INTRUSION THREAT LOGS</h3>"))
    display(df_alerts[["Timestamp", "Signature", "Severity", "Src_IP", "Dest_Port"]])

    print("\nGenerating threat metrics data plots...")
    plt.figure(figsize=(10, 5))
    df_alerts["Signature"].value_counts().plot(kind="barh", color=["#d9534f", "#f0ad4e", "#5bc0de", "#5cb85c"])
    plt.title("NIDS Alert Incident Frequency Profile - Vector Classification Matrix", fontsize=14, fontweight="bold")
    plt.xlabel("Total Dropped / Logged Attack Packets Count")
    plt.ylabel("Triggered Suricata Signature Definition Block")
    plt.tight_layout()
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()

print("\n" + "="*80)
print("     [✔] END OF PIPELINE RUNTIME: METRICS VERIFIED COMPLETELY")
print("="*80)


 >>> STEP 1: INSTALLING SURICATA NIDS ENGINE AND DEPENDENCIES <<<
[*] Syncing package managers and compiling system tools...
[+] Engine core online: This is Suricata version 6.0.4 RELEASE

 >>> STEP 2: INJECTING MALICIOUS THREAT DETECTION RULES <<<
[+] Custom rule signatures successfully written to: /etc/suricata/rules/local.rules
[*] Calibrating configuration matrix paths...
[+] Config engine synced: Active rule validation target locked.

 >>> STEP 3: INITIALIZING MOTOR INTERFACE FOR TRAFFIC SENSING <<<
[*] Launching Suricata NIDS background runtime daemon tracking loopback interface (lo)...
[+] Suricata Engine active under PID: 8031. Actively sniffing traffic...

 >>> STEP 4: SIMULATING TRAFFIC EXPLOITS & RESPONSE INGESTION <<<
[*] Generating adversarial packet variants over loopback adapter link...

[!] Injecting Vector 1: Sending SQL Injection string...
[!] Injecting Vector 2: Sending XSS Payload...
[!] Injecting Vector 3: Accessing Restricted File Paths (/etc/passwd)...
[!] Injec